# How the DataAnalyst Agent works: a hands-on walkthrough

This notebook builds the whole system up one layer at a time, using the project's **real code**
on the **real Olist data**. Run the cells top to bottom (Shift+Enter). No API key is needed: the
agent section uses a *scripted* model so you can see exactly what the AI sends and receives.
An optional last section asks a real question through Gemini's free tier.

**What we are building (and what we are not)**

- We are **not training an LLM**. We use an existing model (Claude or Gemini) as-is.
- It is **not classic RAG** either. RAG = search a pile of documents for relevant text and paste
  it into the prompt. Here the facts live in a *database*, so instead of searching documents the
  model gets **tools**: it can look at the schema and run SQL, then answer from the results.
- This pattern is called a **tool-using agent** (here: an agentic *text-to-SQL* analyst).
  We do give the model some fixed context up front (table list and business definitions); that
  part is a little RAG-like, but the answers come from running queries, not from retrieved text.

**The layers** (each section below is one of them):

```
 your question
      │
      ▼
 5. Agent loop  ──► the model picks a tool ──► our code runs it ──► result goes back ──┐
      ▲                                                                                │
      └────────────────────────── repeat until submit_answer ◄─────────────────────────┘
      │ uses
      ├── 4. Semantic layer   (metrics.yaml: what "revenue" means)
      ├── 3. Catalog          (what's in the data: profiles, joins)
      ├── 2. SQL guard        (only safe, read-only SELECTs get through)
      └── 1. Connectors       (one way to talk to SQLite, Postgres; MongoDB later)
 6. Evals: golden questions with known answers check the whole thing
```

## 0. Setup
Point Python at the project and the Olist data.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
OLIST = ROOT / "data" / "olist" / "olist.sqlite"
print("project:", ROOT)
print("Olist database found:", OLIST.exists())

project: /Users/dev/Projects/DataAnalyst_AI_Agent
Olist database found: True


## 1. Connectors: one interface for every data source

Every data source (SQLite, Postgres; MongoDB later) is a `Connector` with the same two
methods: `get_schema()` and `run_query()`. Everything above this layer only ever calls those two,
so it doesn't care where the data lives. (Files: `src/analyst/connectors/`.)

The connection is opened **read-only** (`mode=ro`): even a buggy query can't change the data.

In [2]:
from analyst.config import QueryLimits
from analyst.connectors import SqlAlchemyConnector

olist = SqlAlchemyConnector(
    "olist",
    f"sqlite:///file:{OLIST}?mode=ro&uri=true",
    limits=QueryLimits(default_row_limit=10, max_row_limit=1000, timeout_seconds=30),
)
schema = olist.get_schema()
print(len(schema.tables), "tables:", [t.name for t in schema.tables])
print()
print("orders columns:", [c.name for c in schema.table("orders").columns])

11 tables: ['customers', 'geolocation', 'leads_closed', 'leads_qualified', 'order_items', 'order_payments', 'order_reviews', 'orders', 'product_category_name_translation', 'products', 'sellers']

orders columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [3]:
result = olist.run_query(
    "SELECT order_status, COUNT(*) AS orders FROM orders GROUP BY 1 ORDER BY orders DESC"
)
print("SQL actually executed:", result.sql)
print("columns:", result.columns)
for row in result.rows:
    print("  ", row)
print("truncated:", result.truncated, "| took", result.elapsed_ms, "ms")

SQL actually executed: SELECT order_status, COUNT(*) AS orders FROM orders GROUP BY 1 ORDER BY orders DESC LIMIT 11
columns: ['order_status', 'orders']
   ['delivered', 96478]
   ['shipped', 1107]
   ['canceled', 625]
   ['unavailable', 609]
   ['invoiced', 314]
   ['processing', 301]
   ['created', 5]
   ['approved', 2]
truncated: False | took 49.29 ms


Notice the executed SQL got a `LIMIT` added: `run_query` always caps rows (here 10, plus one
extra row so it can tell you the result was `truncated`).

## 2. The SQL guard: why the AI can't break anything

Before any SQL reaches the database, `sql_guard.validate_select` **parses it into a tree**
(with the `sqlglot` library) and checks the tree, rather than searching the text for bad words
(which is easy to trick). Rules: exactly one statement, SELECT only, only known tables, no
functions that read files, and a row cap. (File: `src/analyst/sql_guard.py`.)

In [4]:
from analyst.sql_guard import UnsafeQueryError, validate_select

tables = olist.allowed_tables()
tests = [
    "SELECT COUNT(*) FROM orders",
    "DELETE FROM orders",
    "SELECT 1; DROP TABLE orders",
    "SELECT * FROM secret_salaries",
    "WITH x AS (DELETE FROM orders RETURNING *) SELECT * FROM x",
]
for sql in tests:
    try:
        safe = validate_select(sql, dialect="sqlite", max_rows=100, allowed_tables=tables)
        print("ALLOWED ", sql, "\n        ->", safe)
    except UnsafeQueryError as e:
        print("BLOCKED ", sql, "\n        ->", e)

ALLOWED  SELECT COUNT(*) FROM orders 
        -> SELECT COUNT(*) FROM orders LIMIT 100
BLOCKED  DELETE FROM orders 
        -> only SELECT queries are allowed, got DELETE
BLOCKED  SELECT 1; DROP TABLE orders 
        -> expected exactly one statement, got 2
BLOCKED  SELECT * FROM secret_salaries 
        -> unknown table: secret_salaries
BLOCKED  WITH x AS (DELETE FROM orders RETURNING *) SELECT * FROM x 
        -> DELETE is not allowed


**Try it:** add your own sneaky query to the list above and re-run. Even if something slipped
past the guard, the read-only connection from section 1 is a second wall.

## 3. The catalog: what the AI knows about the data before it writes SQL

The model can't guess that `customer_unique_id` (not `customer_id`) identifies a person, or that
`order_items.order_id` joins to `orders.order_id`. The catalog works this out by **profiling**
every column with plain SQL (null share, distinct values, min/max, examples) and **inferring
joins** (Olist declares none). (Files: `src/analyst/catalog/`.)

In [5]:
from analyst.catalog.profiler import profile_table
from analyst.config import ProfilingSettings

row_count, profiles = profile_table(olist, schema.table("customers"), ProfilingSettings())
print("customers rows:", row_count)
for name, p in profiles.items():
    print(f"  {name:26} nulls={p.null_count:<4} distinct={p.distinct_count:<6} "
          f"unique={str(p.is_unique):5} samples={p.samples[:2]}")

customers rows: 99441
  customer_id                nulls=0    distinct=99441  unique=True  samples=['06b8999e2fba1a1fbc88172c00ba8bc7', '18955e83d337fd6b2def6b18a428ac77']
  customer_unique_id         nulls=0    distinct=96096  unique=False samples=['861eff4711a542e4b93843c6dd7febb0', '290c77bc529b7ac935b93aa66c333dc3']
  customer_zip_code_prefix   nulls=0    distinct=14994  unique=False samples=[14409, 9790]
  customer_city              nulls=0    distinct=4119   unique=False samples=['franca', 'sao bernardo do campo']
  customer_state             nulls=0    distinct=27     unique=False samples=['SP', 'SC']


`customer_id` is **unique** (one per order) but `customer_unique_id` is **not** (one person,
several orders). That single fact is the difference between "99,441 customers" (wrong) and
"96,096 customers" (right). Now the full catalog with joins (takes ~10 seconds):

In [6]:
import tempfile
from analyst.catalog import CatalogService, CatalogStore
from analyst.config import Settings, SourceConfig
from analyst.connectors import SourceRegistry

settings = Settings(
    state_dir=Path(tempfile.mkdtemp()),  # private scratch state for this notebook
    sources=[SourceConfig(name="olist", type="sqlite", path=OLIST)],
)
registry = SourceRegistry(settings, only={"olist"})
catalog_service = CatalogService(registry, CatalogStore(settings.app_db_path), settings.profiling)
catalog = catalog_service.refresh("olist")

print("joins the catalog worked out:")
for r in catalog.relationships:
    print(f"  {r.from_table}.{r.from_column}  ->  {r.to_table}.{r.to_column}")

joins the catalog worked out:
  leads_closed.mql_id  ->  leads_qualified.mql_id
  leads_closed.seller_id  ->  sellers.seller_id
  order_items.order_id  ->  orders.order_id
  order_items.product_id  ->  products.product_id
  order_items.seller_id  ->  sellers.seller_id
  order_payments.order_id  ->  orders.order_id
  order_reviews.order_id  ->  orders.order_id
  orders.customer_id  ->  customers.customer_id
  products.product_category_name  ->  product_category_name_translation.product_category_name


## 4. The semantic layer: definitions beat guesses

"Revenue" could mean item price, price + freight, or what customers paid. Each gives a different
number, and none is wrong in general; it's a **business decision**. You approved the Olist
definitions, and they live in `metrics.yaml`. The agent is told to use them.

In [7]:
from analyst.semantic import load_semantics

semantics = load_semantics(ROOT / "metrics.yaml")["olist"]
print(semantics.render()[:1500], "...")

- valid_order: An order whose status is not 'canceled' or 'unavailable'. Revenue, AOV, repeat customers and seller metrics only count valid orders.
  SQL: orders.order_status NOT IN ('canceled', 'unavailable')
- revenue: Sum of item price over valid orders. Excludes freight_value. Not payment_value, which also includes freight, installment interest and vouchers.
  SQL: SUM(order_items.price)
- order_date: orders.order_purchase_timestamp. Months are calendar months of that timestamp.
  SQL: strftime('%Y-%m', orders.order_purchase_timestamp)
- aov: Average order value = revenue / number of distinct valid orders with items.
  SQL: SUM(order_items.price) / COUNT(DISTINCT orders.order_id)
- delivered_order: order_status = 'delivered' and order_delivered_customer_date is not null.
  SQL: orders.order_status = 'delivered' AND orders.order_delivered_customer_date IS NOT NULL
- late_delivery: A delivered order whose delivery day is after the estimated delivery day. Delivered on the estimated da

## 5. The agent: how tool calling actually works

This is the part people find surprising. **The model never runs anything itself.** Each turn:

1. We send the model: a *system prompt* (role, tables, definitions), the conversation so far,
   and a list of **tools** it may ask for (`list_tables`, `describe_table`, `run_sql`,
   `submit_answer`).
2. The model replies with a **request to call a tool** (a name and JSON arguments).
3. **Our code** runs that tool (through the connector and guard) and sends the result back.
4. Repeat, until the model calls `submit_answer` with its answer and final SQL.
5. Our code runs that final SQL itself and keeps the result: the answer is backed by a real query.

First, here is the system prompt the model receives:

In [8]:
from analyst.agent.prompt import build_system_prompt

system_prompt = build_system_prompt(catalog, semantics)
print(system_prompt[:2500], "...")

You are a data analyst answering business questions about one database. Work from the data:
look at the schema, run queries to check values and joins, and base every number you report
on a query you ran. When you are done, call submit_answer with a short answer and the one
SELECT that produces it.

Database: olist (sqlite SQL). Queries are read-only and row-limited.

Tables:
- customers (99441 rows): customer_id TEXT, customer_unique_id TEXT, customer_zip_code_prefix INTEGER, customer_city TEXT, customer_state TEXT
- geolocation (1000163 rows): geolocation_zip_code_prefix INTEGER, geolocation_lat REAL, geolocation_lng REAL, geolocation_city TEXT, geolocation_state TEXT
- leads_closed (842 rows): mql_id TEXT, seller_id TEXT, sdr_id TEXT, sr_id TEXT, won_date TEXT, business_segment TEXT, lead_type TEXT, lead_behaviour_profile TEXT, has_company INTEGER, has_gtin INTEGER, average_stock TEXT, business_type TEXT, declared_product_catalog_size REAL, declared_monthly_revenue REAL
- leads_quali

And the tools, exactly as the model sees them (name, description, input schema):

In [9]:
from analyst.agent import TOOL_DEFINITIONS

for tool in TOOL_DEFINITIONS:
    print(f"{tool['name']}: {tool['description'][:90]}...")
    print("   inputs:", list(tool["input_schema"]["properties"]) or "none")

list_tables: List every table with its row count, column count and description....
   inputs: none
describe_table: Show a table's columns (type, null share, distinct count, min/max, sample values, descript...
   inputs: ['table']
run_sql: Run one read-only SELECT and see the result. Use it to explore values and to check your fi...
   inputs: ['sql']
submit_answer: Finish: give the answer for the user and the one query that produces it. Call this exactly...
   inputs: ['answer', 'sql', 'answerable']


Now a full run. Instead of a real model we use a **scripted** one: it replies with pre-written
tool requests, the way a real model would. Everything else (tools, guard, database, the loop)
is the real code. This is also how the project's tests work, so they're free and repeatable.

In [10]:
import itertools
from types import SimpleNamespace

from analyst.agent import Agent
from analyst.config import AgentSettings

ids = itertools.count(1)

def wants_tool(name, **arguments):
    """What a model reply looks like when it asks for a tool."""
    block = SimpleNamespace(type="tool_use", id=f"call_{next(ids)}", name=name, input=arguments)
    usage = SimpleNamespace(input_tokens=0, output_tokens=0, cache_read_input_tokens=0,
                            cache_creation_input_tokens=0)
    return SimpleNamespace(content=[block], stop_reason="tool_use", usage=usage,
                           stop_details=None)

LATE_SQL = '''SELECT ROUND(100.0 * SUM(date(order_delivered_customer_date)
                          > date(order_estimated_delivery_date)) / COUNT(*), 2) AS late_pct
FROM orders
WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL'''

class ScriptedModel:
    """Pretends to be the model: returns these replies in order and prints the exchange."""
    def __init__(self, replies):
        self.replies = list(replies)
        self.beta = SimpleNamespace(messages=SimpleNamespace(create=self.create))

    def create(self, **request):
        last = request["messages"][-1]
        if isinstance(last["content"], list):  # tool results coming back from our code
            for block in last["content"]:
                print("   our code ->", block["content"][:160].replace("\n", " "), "...")
        reply = self.replies.pop(0)
        call = reply.content[0]
        print(f"\nmodel asks: {call.name}({json.dumps(call.input)[:100]})")
        return reply

model = ScriptedModel([
    wants_tool("list_tables"),
    wants_tool("describe_table", table="orders"),
    wants_tool("run_sql", sql="SELECT order_status, COUNT(*) FROM orders GROUP BY 1"),
    wants_tool("submit_answer", answer="6.77% of delivered orders arrived late.",
               sql=LATE_SQL, answerable=True),
])
agent = Agent(model, olist, catalog, AgentSettings(), semantics)
answer = agent.ask("What is our late-delivery rate?")

print("\n=== final ===")
print("status:", answer.status)
print("answer:", answer.answer)
print("result (computed by our code from the final SQL):", answer.result.rows)
print("tool calls recorded:", [s.tool for s in answer.steps])


model asks: list_tables({})
   our code -> [{"table": "customers", "rows": 99441, "columns": 5}, {"table": "geolocation", "rows": 1000163, "columns": 5}, {"table": "leads_closed", "rows": 842, "columns": ...

model asks: describe_table({"table": "orders"})
   our code -> {"table": "orders", "rows": 99441, "columns": [{"name": "order_id", "type": "TEXT", "null_share": 0.0, "distinct": 99441, "min": "00010242fe8c5a6d1ba2dd792cb162 ...

model asks: run_sql({"sql": "SELECT order_status, COUNT(*) FROM orders GROUP BY 1"})
   our code -> {"columns": ["order_status", "COUNT(*)"], "rows": [["approved", 2], ["canceled", 625], ["created", 5], ["delivered", 96478], ["invoiced", 314], ["processing", 3 ...

model asks: submit_answer({"answer": "6.77% of delivered orders arrived late.", "sql": "SELECT ROUND(100.0 * SUM(date(order_de)

=== final ===
status: answered
answer: 6.77% of delivered orders arrived late.
result (computed by our code from the final SQL): [[6.77]]
tool calls recorded: ['list_

**Try it:** change the `run_sql` step above to `"DELETE FROM orders"` and re-run. The guard
blocks it, our code sends the error back to the "model" as a tool result, and the loop carries on.

## 6. Evals: how we know the answers are right

The golden set (`evals/golden.yaml`) has 30 questions with answers computed from SQL you
reviewed. The scorer compares the agent's result table with the golden one, tolerating harmless
differences (extra columns, column order, 6.77 vs 0.0677, rounding) but not wrong numbers.

In [11]:
import yaml
from analyst.evals.scoring import answer_match

golden = yaml.safe_load((ROOT / "evals" / "golden.yaml").read_text())
question = next(q for q in golden["questions"] if q["id"] == "del_late_rate")
print("question:", question["question"])
print("golden answer:", question["expected"]["rows"])
print("agent answer: ", answer.result.rows)
print("score:", answer_match(answer.result.columns, answer.result.rows,
                             question["expected"]["rows"], ordered=False))
print()
print("a wrong answer:", answer_match(["late_pct"], [[8.11]], question["expected"]["rows"],
                                       ordered=False))

question: What is our late-delivery rate?
golden answer: [[6.77]]
agent answer:  [[6.77]]
score: (True, 'matched on columns: late_pct')

a wrong answer: (False, 'no column matches golden column 1 ([6.77]...)')


## 7. Optional: ask a real model (free, Gemini)

Everything above used a scripted model. This cell asks the real one, using the provider set in
`config.yaml` (Gemini's free tier, key `GEMINI_API_KEY` in `.env`). It can take a minute when
Google's free tier is busy. Set `RUN_LIVE = True` to try it.

In [12]:
RUN_LIVE = False

if RUN_LIVE:
    from dotenv import load_dotenv
    from analyst.agent.providers import make_llm_client
    from analyst.config import load_settings

    load_dotenv(ROOT / ".env")
    live = load_settings(ROOT / "config.yaml").agent
    real_agent = Agent(make_llm_client(live), olist, catalog, live, semantics)
    reply = real_agent.ask("Which 3 states have the most customers?")
    print("status:", reply.status, "| turns:", reply.turns)
    print("answer:", reply.answer)
    print("sql:", reply.sql)
    print("result:", reply.result.rows if reply.result else None)
    for step in reply.steps:
        print("  step:", step.tool, "ok" if step.ok else f"error: {step.error}")
else:
    print("Set RUN_LIVE = True to ask Gemini for real.")

Set RUN_LIVE = True to ask Gemini for real.


## Recap: where everything lives

| Layer | File | One-line job |
|---|---|---|
| Connectors | `src/analyst/connectors/` | one `get_schema` / `run_query` for every source |
| SQL guard | `src/analyst/sql_guard.py` | only one safe, capped SELECT gets through |
| Catalog | `src/analyst/catalog/` | profiles columns, infers joins, stores descriptions |
| Semantic layer | `metrics.yaml`, `src/analyst/semantic.py` | business definitions the agent must use |
| Agent | `src/analyst/agent/` | tools + loop + prompt; providers for Claude or Gemini |
| API | `src/analyst/api/app.py` | HTTP endpoints, including `POST /sources/{source}/ask` |
| Evals | `evals/golden.yaml`, `src/analyst/evals/` | known answers and the scorer |

**Not built yet:** memory of past answers (phase 4) and MongoDB (phase 5).

In [13]:
registry.close()
olist.close()